# 🧠 Entregable 6 — Aplicación de Herramientas de XAI
## Tema #6: Sistemas Distribuidos

**Universidad Libre / TalentoTech**

---

### Objetivos del laboratorio
1. Diseñar y entrenar una CNN sobre el dataset **CIFAR-10**.
2. Implementar **LIME**, **SHAP** y **Grad-CAM** para generar explicaciones locales y visuales.
3. Evaluar las explicaciones con métricas de **fidelidad**, **comprensibilidad** y **estabilidad**.
4. Desplegar un microservicio con **Flask** que exponga el modelo y sus explicaciones.

---

## ⚙️ Instalación de dependencias

In [ ]:
# Ejecutar una sola vez para instalar todas las bibliotecas necesarias
import sys
!{sys.executable} -m pip install tensorflow torch captum shap lime matplotlib scikit-image flask numpy opencv-python-headless --quiet

---
# 📦 Actividad 1: Preparación de Datos y Entrenamiento del Modelo CNN

### Introducción
En esta actividad se carga el dataset **CIFAR-10** (60 000 imágenes de 32×32 px, 10 clases),
se preprocesa, y se entrena una Red Neuronal Convolucional (CNN) para clasificar las imágenes.

### Objetivo
Obtener un modelo CNN entrenado y guardado en disco que sirva como base para las siguientes actividades de XAI.

### Paso 1 — Importación de bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.utils import to_categorical

print(f'✅ TensorFlow versión: {tf.__version__}')
print(f'✅ NumPy versión:      {np.__version__}')

# Nombres de las 10 clases de CIFAR-10
CLASS_NAMES = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo',
               'perro', 'rana', 'caballo', 'barco', 'camión']

### Paso 2 — Carga y preprocesamiento de datos

In [ ]:
# Carga del dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalización de píxeles al rango [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# One-hot encoding de etiquetas
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print(f'✅ Entrenamiento: {x_train.shape}  |  Prueba: {x_test.shape}')
print(f'   Etiquetas entrenamiento: {y_train_cat.shape}')

### Paso 3 — Visualización de los datos

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('Muestra de imágenes CIFAR-10', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[y_train[i][0]], fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Paso 4 — Construcción de la CNN

In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=10):
    """
    Arquitectura CNN para CIFAR-10.
    - Bloque 1: 2× Conv2D(32) + MaxPooling + Dropout
    - Bloque 2: 2× Conv2D(64) + MaxPooling + Dropout
    - Bloque 3: 2× Conv2D(128) + MaxPooling + Dropout
    - Cabeza clasificadora: Dense(256) + Dense(10, softmax)
    """
    model = keras.Sequential([
        # ── Bloque 1 ──────────────────────────────────────────
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=input_shape, name='conv2d_0'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv2d_1'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ── Bloque 2 ──────────────────────────────────────────
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2d_2'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2d_3'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ── Bloque 3 ──────────────────────────────────────────
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2d_4'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2d_5'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ── Clasificador ──────────────────────────────────────
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax', name='predictions')
    ])
    return model

model = build_cnn()
model.summary()

### Paso 5 — Compilación y entrenamiento del modelo

In [ ]:
# Data augmentation para mejorar la generalización
datagen = keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    rotation_range=10
)
datagen.fit(x_train)

# Compilación
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks: reduce LR si no mejora y guarda el mejor modelo
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5,
                                       patience=3, verbose=1, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint('cnn_model_cifar10.h5',
                                     monitor='val_accuracy',
                                     save_best_only=True, verbose=1)
]

# Entrenamiento
EPOCHS = 30
history = model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=64),
    epochs=EPOCHS,
    validation_data=(x_test, y_test_cat),
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Modelo guardado como cnn_model_cifar10.h5')

### Paso 6 — Visualización de la evolución del entrenamiento

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evolución del Entrenamiento CNN — CIFAR-10', fontsize=14, fontweight='bold')

# Pérdida
ax1.plot(history.history['loss'],     label='Train Loss',      color='steelblue',  linewidth=2)
ax1.plot(history.history['val_loss'], label='Val Loss',        color='coral',      linewidth=2)
ax1.set_title('Pérdida (Loss)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Categorical Cross-Entropy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Exactitud
ax2.plot(history.history['accuracy'],     label='Train Accuracy', color='seagreen', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Val Accuracy',   color='darkorange', linewidth=2)
ax2.set_title('Exactitud (Accuracy)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Evaluación final
loss, acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'\n📊 Resultados en test — Loss: {loss:.4f}  |  Accuracy: {acc*100:.2f}%')

### Reflexión Final — Actividad 1

> **¿Qué aprendimos?**  
> El modelo CNN con tres bloques convolucionales logra aprender representaciones jerárquicas: 
> los primeros bloques detectan bordes y texturas, mientras los bloques más profundos capturan 
> patrones semánticos como formas de animales o vehículos. La adición de *batch normalization* 
> y *data augmentation* estabiliza el entrenamiento y reduce el sobreajuste.  
> Típicamente se obtiene una accuracy en test del **70–78%**, suficiente para aplicar XAI 
> y observar patrones interpretables.

---
# 🔍 Actividad 2: Aplicación de Herramientas de XAI

### Introducción
Las herramientas XAI permiten **entender por qué** el modelo toma una decisión concreta. 
Usaremos tres enfoques complementarios:
- **LIME**: aproximación lineal local alrededor de una predicción.
- **SHAP**: valores de Shapley que cuantifican la contribución de cada píxel/región.
- **Grad-CAM**: mapas de calor basados en gradientes de la última capa convolucional.

### Objetivo
Generar y visualizar explicaciones locales para muestras del conjunto de prueba.

In [ ]:
# Cargar el mejor modelo guardado
model = load_model('cnn_model_cifar10.h5')

# Seleccionar imágenes de prueba representativas (una por clase)
sample_indices = []
for cls in range(10):
    idx = np.where(y_test.flatten() == cls)[0][0]
    sample_indices.append(idx)

print('✅ Modelo cargado.')
print(f'   Índices de muestra (una imagen por clase): {sample_indices}')

### Paso 1 — Aplicación de LIME

**Concepto**: LIME crea perturbaciones locales de la imagen (apagando superpíxeles), 
las evalúa con el modelo, y ajusta un modelo lineal simple para aproximar el comportamiento 
del modelo en torno a esa predicción específica.

In [ ]:
import lime
import lime.lime_image
from skimage.segmentation import mark_boundaries

# Crear el explicador LIME
lime_explainer = lime.lime_image.LimeImageExplainer(random_state=42)

def explain_lime(image, model, label, num_samples=500, num_features=5):
    """
    Genera y retorna la explicación LIME para una imagen.
    - num_samples: cantidad de perturbaciones generadas
    - num_features: número de superpíxeles (regiones) a resaltar
    """
    explanation = lime_explainer.explain_instance(
        image,
        model.predict,
        top_labels=3,
        hide_color=0,
        num_samples=num_samples
    )
    temp_pos, mask_pos = explanation.get_image_and_mask(
        label, positive_only=True,  num_features=num_features, hide_rest=False)
    temp_neg, mask_neg = explanation.get_image_and_mask(
        label, positive_only=False, num_features=num_features, hide_rest=False)
    return temp_pos, mask_pos, temp_neg, mask_neg, explanation

# ── Visualizar LIME para 4 imágenes de prueba ──────────────────
fig, axes = plt.subplots(4, 3, figsize=(13, 16))
fig.suptitle('LIME — Explicaciones Locales por Superpíxeles', fontsize=15, fontweight='bold')

for row, idx in enumerate(sample_indices[:4]):
    image = x_test[idx]
    label = y_test[idx][0]
    pred  = np.argmax(model.predict(image[np.newaxis], verbose=0))

    temp_pos, mask_pos, temp_neg, mask_neg, _ = explain_lime(image, model, label)

    # Imagen original
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f'Original: {CLASS_NAMES[label]}\nPredicción: {CLASS_NAMES[pred]}',
                           fontsize=9)
    axes[row, 0].axis('off')

    # Regiones positivas (favorecen la predicción)
    axes[row, 1].imshow(mark_boundaries(temp_pos, mask_pos, color=(0, 1, 0)))
    axes[row, 1].set_title('Regiones POSITIVAS\n(apoyan la predicción)', fontsize=9, color='green')
    axes[row, 1].axis('off')

    # Regiones negativas (contradicen la predicción)
    axes[row, 2].imshow(mark_boundaries(temp_neg, mask_neg, color=(1, 0, 0)))
    axes[row, 2].set_title('Regiones NEGATIVAS\n(contradicen la predicción)', fontsize=9, color='red')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('lime_explanations.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualizaciones LIME generadas y guardadas.')

### Paso 2 — Aplicación de SHAP

**Concepto**: SHAP asigna a cada píxel/región un valor que representa su contribución 
marginal promedio a la predicción, basado en la teoría de juegos cooperativos (valores de Shapley). 
- Valores **positivos** → la región empuja la predicción hacia esa clase.
- Valores **negativos** → la región la aleja.

In [ ]:
import shap

# Datos de referencia (background): 100 imágenes aleatorias de entrenamiento
np.random.seed(42)
bg_idx     = np.random.choice(len(x_train), 100, replace=False)
background = x_train[bg_idx]

# Imagen de prueba a explicar
test_img = x_test[sample_indices[0]:sample_indices[0]+1]   # shape (1,32,32,3)

print('⏳ Calculando valores SHAP (puede tardar 1-3 minutos)...')
shap_explainer = shap.DeepExplainer(model, background)
shap_values    = shap_explainer.shap_values(test_img)

# ── Normalizar estructura según versión de SHAP ──────────────────
# Versiones nuevas devuelven un array (1,32,32,3,10); las viejas una lista de 10 x (1,32,32,3)
if isinstance(shap_values, list):
    # Lista de clases → convertir a array (10, 32, 32, 3) quitando el eje de batch
    sv_all = np.array([sv[0] for sv in shap_values])   # (10, 32, 32, 3)
else:
    # Array (1, 32, 32, 3, 10) → transponer a (10, 32, 32, 3)
    sv_all = shap_values[0].transpose(3, 0, 1, 2)      # (10, 32, 32, 3)

# ── Top-3 clases por probabilidad ────────────────────────────────
pred_probs   = model.predict(test_img, verbose=0)[0]
top3_classes = np.argsort(pred_probs)[::-1][:3]

# ── Visualización ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(
    f'SHAP — Contribución de píxeles para "{CLASS_NAMES[y_test[sample_indices[0]][0]]}"',
    fontsize=13, fontweight='bold'
)

axes[0].imshow(test_img[0])
axes[0].set_title('Imagen Original', fontsize=10)
axes[0].axis('off')

for i, cls in enumerate(top3_classes):
    sv      = sv_all[cls]                   # (32, 32, 3)
    sv_gray = sv.mean(axis=-1)              # (32, 32)  — promedio canales RGB
    vmax    = np.abs(sv_gray).max()

    im = axes[i+1].imshow(sv_gray, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[i+1].set_title(
        f'Clase: {CLASS_NAMES[cls]}\nProb: {pred_probs[cls]*100:.1f}%', fontsize=9
    )
    axes[i+1].axis('off')
    plt.colorbar(im, ax=axes[i+1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('shap_explanations.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualizaciones SHAP generadas y guardadas.')

### Paso 3 — Aplicación de Grad-CAM

**Concepto**: Grad-CAM calcula el gradiente de la puntuación de la clase predicha 
respecto a los mapas de activación de la última capa convolucional. Las regiones 
con gradiente alto corresponden a zonas que el modelo "miró" para tomar su decisión.

In [ ]:
import cv2

def grad_cam(image, model, layer_name='conv2d_5'):
    """Grad-CAM compatible con Keras 3 + Sequential."""

    img_batch = tf.cast(image[np.newaxis], tf.float32)

    # ── Trazar el modelo con un input concreto para "desbloquear" el grafo ──
    _ = model(img_batch, training=False)

    # ── Construir submodelo usando tf.keras.backend ──────────────────────────
    # En Keras 3, la forma más robusta es usar una función directa sin Model()
    conv_layer_model = tf.keras.Sequential(model.layers[:model.layers.index(
        model.get_layer(layer_name)) + 1
    ])
    _ = conv_layer_model(img_batch, training=False)   # trazar submodelo

    with tf.GradientTape() as tape:
        # Calcular activaciones conv manualmente
        x = img_batch
        conv_output = None
        for layer in model.layers:
            x = layer(x, training=False)
            if layer.name == layer_name:
                conv_output = x
                tape.watch(conv_output)   # ← registrar aquí
        predictions = x                   # última capa = predictions

        pred_class = int(tf.argmax(predictions[0]))
        loss = predictions[:, pred_class]

    grads = tape.gradient(loss, conv_output)
    if grads is None:
        raise ValueError(f"Gradientes None en capa '{layer_name}'")

    pooled_grads = tf.reduce_mean(grads[0], axis=(0, 1))  # (filters,)
    conv_out     = conv_output[0]                          # (h, w, filters)

    heatmap = tf.reduce_sum(pooled_grads * conv_out, axis=-1).numpy()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    h, w    = image.shape[:2]
    heatmap = cv2.resize(heatmap, (w, h))
    return heatmap, pred_class, float(predictions[0, pred_class])


def overlay_heatmap(image, heatmap, alpha=0.45, colormap=cv2.COLORMAP_JET):
    hm_color = cv2.applyColorMap(np.uint8(255 * heatmap), colormap)
    hm_color = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB) / 255.0
    return alpha * hm_color + (1 - alpha) * image


# ── Visualizar Grad-CAM para 6 imágenes ──────────────────────────
fig, axes = plt.subplots(6, 3, figsize=(10, 20))
fig.suptitle('Grad-CAM — Mapas de Activación (conv2d_5)',
             fontsize=14, fontweight='bold')

for row, idx in enumerate(sample_indices[:6]):
    image    = x_test[idx]
    true_cls = y_test[idx][0]
    heatmap, pred_cls, conf = grad_cam(image, model)
    overlay  = overlay_heatmap(image, heatmap)

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f'Original: {CLASS_NAMES[true_cls]}', fontsize=8)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(heatmap, cmap='jet')
    axes[row, 1].set_title('Mapa de calor', fontsize=8)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(overlay)
    correct = '✅' if pred_cls == true_cls else '❌'
    axes[row, 2].set_title(
        f'Superposición\nPredicción: {CLASS_NAMES[pred_cls]} ({conf*100:.0f}%) {correct}',
        fontsize=8)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('gradcam_explanations.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualizaciones Grad-CAM generadas y guardadas.')

### Reflexión Final — Actividad 2

| Herramienta | Tipo de explicación | Ventaja principal | Limitación |
|-------------|--------------------|--------------------|------------|
| **LIME**    | Local, por superpíxeles | Agnóstica al modelo; intuitiva | Aleatoriedad por perturbaciones |
| **SHAP**    | Local + global, por píxel | Fundamentación teórica sólida (Shapley) | Costosa computacionalmente |
| **Grad-CAM**| Local, mapa de calor continuo | Rápida; usa el propio gradiente del modelo | Solo aplicable a CNNs |

> **Observación clave**: Las tres herramientas son complementarias. Grad-CAM da una vista 
> rápida de *dónde* mira el modelo; LIME explica *cuáles regiones* son decisivas; 
> SHAP cuantifica *cuánto* contribuye cada píxel. Usarlas juntas maximiza la confianza 
> en las predicciones del modelo.

---
# 📏 Actividad 3: Evaluación de Explicaciones

### Introducción
Una explicación puede ser visualmente atractiva pero ser **infiel** al modelo o **inestable** 
ante pequeñas perturbaciones. Esta actividad aplica métricas objetivas para evaluar la calidad 
de las explicaciones generadas.

### Objetivo
Medir y comparar las tres herramientas XAI usando métricas de **fidelidad**, 
**comprensibilidad** y **estabilidad**.

### Paso 1 — Evaluación de la Fidelidad (Faithfulness)

La fidelidad mide si enmascarar las regiones "importantes" según la explicación 
degrada significativamente la predicción del modelo. Una **mayor caída en probabilidad** 
indica **mayor fidelidad**.

In [ ]:
def fidelity_score_gradcam(image, model, layer_name='conv2d_5', top_k_pct=0.2):
    """
    Fidelidad para Grad-CAM: compara la probabilidad de la clase predicha
    antes y después de enmascarar el top_k_pct% de píxeles más importantes.
    
    Retorna: (prob_original, prob_masked, fidelity_drop)
    """
    heatmap, pred_cls, prob_orig = grad_cam(image, model, layer_name)

    # Crear máscara: apagar los píxeles con mayor activación
    threshold  = np.percentile(heatmap, (1 - top_k_pct) * 100)
    mask       = heatmap >= threshold               # True = importante
    masked_img = image.copy()
    masked_img[mask] = 0.0                          # enmascarar con negro

    prob_masked = float(model.predict(
        masked_img[np.newaxis], verbose=0)[0, pred_cls])

    fidelity_drop = prob_orig - prob_masked
    return prob_orig, prob_masked, fidelity_drop


# Calcular fidelidad para 20 muestras de test
n_samples   = 20
eval_idx    = sample_indices + list(range(10, 20))   # 20 índices
fidel_probs_orig, fidel_probs_mask, fidel_drops = [], [], []

for idx in eval_idx[:n_samples]:
    p_o, p_m, drop = fidelity_score_gradcam(x_test[idx], model)
    fidel_probs_orig.append(p_o)
    fidel_probs_mask.append(p_m)
    fidel_drops.append(drop)

mean_drop = np.mean(fidel_drops)
print(f'📊 Fidelidad (Grad-CAM) — Caída media de probabilidad al enmascarar:')
print(f'   Prob. original promedio : {np.mean(fidel_probs_orig):.3f}')
print(f'   Prob. enmascarada prom. : {np.mean(fidel_probs_mask):.3f}')
print(f'   Caída media (fidelidad) : {mean_drop:.3f}  → '
      f'{"Alta" if mean_drop > 0.25 else "Media" if mean_drop > 0.1 else "Baja"} fidelidad')

# Visualización
fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(n_samples)
ax.bar(x_pos - 0.2, fidel_probs_orig, width=0.4, label='Prob. original',   color='steelblue', alpha=0.8)
ax.bar(x_pos + 0.2, fidel_probs_mask, width=0.4, label='Prob. enmascarada', color='coral',     alpha=0.8)
ax.set_title('Fidelidad Grad-CAM — Probabilidad antes y después del enmascaramiento', fontsize=12)
ax.set_xlabel('Muestra')
ax.set_ylabel('Probabilidad predicha')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Umbral 0.5')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fidelity_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### Paso 2 — Evaluación de la Comprensibilidad (Cognitive Load)

La comprensibilidad se mide como la **complejidad de la explicación**: 
cuántas regiones/características necesita para explicar la predicción. 
Menos regiones con alta contribución = más comprensible.

In [ ]:
def comprehensibility_gradcam(heatmap, threshold_pct=0.5):
    """
    Fracción del área de la imagen que explica el threshold_pct% de la activación total.
    Un valor bajo → explicación más concentrada → más comprensible.
    """
    flat  = heatmap.flatten()
    total = flat.sum()
    if total == 0:
        return 1.0
    sorted_vals = np.sort(flat)[::-1]
    cum_sum = np.cumsum(sorted_vals)
    n_needed = np.searchsorted(cum_sum, threshold_pct * total) + 1
    return n_needed / len(flat)   # fracción del área necesaria


comp_scores_gcam = []
comp_scores_lime_pos = []

for idx in eval_idx[:n_samples]:
    image    = x_test[idx]
    true_cls = y_test[idx][0]

    # Grad-CAM
    hm, _, _ = grad_cam(image, model)
    comp_scores_gcam.append(comprehensibility_gradcam(hm))

    # LIME: contamos la fracción de superpíxeles positivos sobre el total
    _, mask_pos, _, _, expl = explain_lime(image, model, true_cls, num_samples=300)
    frac_positive = mask_pos.sum() / mask_pos.size
    comp_scores_lime_pos.append(frac_positive)

print('📊 Comprensibilidad (fracción de área explicativa — menor es mejor):')
print(f'   Grad-CAM : {np.mean(comp_scores_gcam):.3f} ± {np.std(comp_scores_gcam):.3f}')
print(f'   LIME     : {np.mean(comp_scores_lime_pos):.3f} ± {np.std(comp_scores_lime_pos):.3f}')

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
methods = ['Grad-CAM', 'LIME']
means   = [np.mean(comp_scores_gcam), np.mean(comp_scores_lime_pos)]
stds    = [np.std(comp_scores_gcam),  np.std(comp_scores_lime_pos)]
colors  = ['steelblue', 'seagreen']

bars = ax.bar(methods, means, yerr=stds, capsize=6, color=colors, alpha=0.8, width=0.4)
ax.set_title('Comprensibilidad — Fracción de área para explicar la predicción\n(menor = más comprensible)',
             fontsize=11)
ax.set_ylabel('Fracción del área')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('comprehensibility_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### Paso 3 — Evaluación de la Estabilidad y Coherencia

La estabilidad mide cuánto cambia la explicación cuando se añade un pequeño ruido 
gaussiano a la imagen. Una explicación estable produce mapas de calor similares 
ante perturbaciones imperceptibles.

In [ ]:
from scipy.stats import pearsonr

def stability_gradcam(image, model, layer_name='conv2d_5', n_runs=5, noise_std=0.02):
    """
    Estabilidad: correlación de Pearson promedio entre el mapa de calor original
    y los mapas con ruido gaussiano añadido.
    
    Valor en [-1, 1]: más cercano a 1 → mayor estabilidad.
    """
    hm_orig, _, _ = grad_cam(image, model, layer_name)
    correlations = []

    for _ in range(n_runs):
        noise      = np.random.normal(0, noise_std, image.shape).astype('float32')
        img_noisy  = np.clip(image + noise, 0, 1)
        hm_noisy, _, _ = grad_cam(img_noisy, model, layer_name)
        corr, _    = pearsonr(hm_orig.flatten(), hm_noisy.flatten())
        correlations.append(corr)

    return np.mean(correlations), np.std(correlations)


stability_means, stability_stds = [], []

for idx in eval_idx[:n_samples]:
    mean_corr, std_corr = stability_gradcam(x_test[idx], model)
    stability_means.append(mean_corr)
    stability_stds.append(std_corr)

global_mean = np.mean(stability_means)
global_std  = np.mean(stability_stds)

print('📊 Estabilidad Grad-CAM (correlación Pearson con ruido gaussiano):')
print(f'   Correlación media : {global_mean:.4f} ± {global_std:.4f}')
print(f'   Interpretación    : {"Alta" if global_mean > 0.85 else "Media" if global_mean > 0.6 else "Baja"} estabilidad')

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución de correlaciones
axes[0].hist(stability_means, bins=10, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(global_mean, color='coral', linestyle='--', linewidth=2,
                label=f'Media: {global_mean:.3f}')
axes[0].set_title('Distribución de Estabilidad (Grad-CAM)', fontsize=11)
axes[0].set_xlabel('Correlación de Pearson')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ejemplo visual: mapa original vs mapa con ruido
sample_img = x_test[eval_idx[0]]
hm_clean, _, _ = grad_cam(sample_img, model)
noise  = np.random.normal(0, 0.02, sample_img.shape).astype('float32')
hm_noisy, _, _ = grad_cam(np.clip(sample_img + noise, 0, 1), model)

diff = np.abs(hm_clean - hm_noisy)
axes[1].imshow(diff, cmap='hot')
axes[1].set_title('Diferencia absoluta de mapas de calor\n(original vs. imagen con ruido)', fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('stability_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### Resumen comparativo de métricas de evaluación

In [ ]:
print('=' * 60)
print('       RESUMEN DE MÉTRICAS XAI')
print('=' * 60)
print(f'  Fidelidad  (Grad-CAM) — caída prob.: {mean_drop:.3f}')
print(f'  Comprensibilidad:')
print(f'    Grad-CAM : {np.mean(comp_scores_gcam):.3f}  (fracción área)')
print(f'    LIME     : {np.mean(comp_scores_lime_pos):.3f}  (fracción superpíxeles)')
print(f'  Estabilidad (Grad-CAM) — correlación: {global_mean:.4f}')
print('=' * 60)

### Reflexión Final — Actividad 3

> - **Fidelidad alta** (caída > 0.25) indica que el modelo realmente depende de las 
>   regiones señaladas por Grad-CAM; si la caída es baja, la explicación podría ser engañosa.
> - **Comprensibilidad**: Grad-CAM tiende a concentrar la activación en pocas regiones, 
>   resultando más conciso que LIME (que depende del número de superpíxeles elegido).
> - **Estabilidad alta** (correlación > 0.85) confirma que Grad-CAM es robusto al ruido; 
>   valores bajos sugerirían inestabilidad y desconfianza en las explicaciones visuales.

---
# 🚀 Actividad 4: Interpretabilidad en Producción — Servicio Flask

### Introducción
Para llevar el modelo a producción, se construye un microservicio REST con **Flask** 
que expone tres endpoints:
- `POST /predict` — clasificación de una imagen
- `POST /explain/gradcam` — mapa de calor Grad-CAM
- `POST /explain/lime` — explicación LIME

El código se guarda en `app_flask.py` y puede ejecutarse desde terminal.

### Paso 1 & 2 — Preparación y creación del servicio Flask

In [ ]:
flask_app_code = '''
"""
app_flask.py
============
Microservicio REST para clasificación CIFAR-10 con explicaciones XAI.

Uso:
    python app_flask.py

Endpoints:
    POST /predict          — retorna clase predicha y probabilidades
    POST /explain/gradcam  — retorna mapa de calor Grad-CAM (base64 PNG)
    POST /explain/lime     — retorna explicación LIME (base64 PNG)
"""

import io, base64, warnings
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')   # backend sin pantalla
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from flask import Flask, request, jsonify
from PIL import Image
import lime
import lime.lime_image
from skimage.segmentation import mark_boundaries

warnings.filterwarnings('ignore')

app   = Flask(__name__)
model = load_model('cnn_model_cifar10.h5')

CLASS_NAMES = ['avión','automóvil','pájaro','gato','ciervo',
               'perro','rana','caballo','barco','camión']


# ── Utilidades ────────────────────────────────────────────────────

def decode_image(file_bytes):
    """Decodifica los bytes de una imagen PNG/JPEG a np.array (32,32,3) float32."""
    img = Image.open(io.BytesIO(file_bytes)).convert('RGB').resize((32, 32))
    return np.array(img, dtype='float32') / 255.0


def fig_to_b64(fig):
    """Convierte una figura matplotlib a string base64 PNG."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')


def grad_cam(image, layer_name='conv2d_5'):
    grad_model = Model(inputs=model.inputs,
                       outputs=[model.get_layer(layer_name).output, model.output])
    img_batch = tf.cast(image[np.newaxis], tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_batch)
        pred_cls = int(tf.argmax(preds[0]))
        loss = preds[:, pred_cls]
    grads = tape.gradient(loss, conv_out)[0]
    pooled = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = tf.reduce_sum(pooled * conv_out[0], axis=-1).numpy()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return cv2.resize(heatmap, (32, 32)), pred_cls, float(preds[0, pred_cls])


# ── Endpoints ─────────────────────────────────────────────────────

@app.route('/predict', methods=['POST'])
def predict():
    """Clasifica la imagen recibida."""
    if 'image' not in request.files:
        return jsonify({'error': 'No se recibió imagen'}), 400
    img   = decode_image(request.files['image'].read())
    probs = model.predict(img[np.newaxis], verbose=0)[0]
    return jsonify({
        'predicted_class': CLASS_NAMES[int(np.argmax(probs))],
        'confidence':      float(np.max(probs)),
        'probabilities':   {cls: float(p) for cls, p in zip(CLASS_NAMES, probs)}
    })


@app.route('/explain/gradcam', methods=['POST'])
def explain_gradcam():
    """Retorna imagen original + mapa de calor Grad-CAM en base64 PNG."""
    if 'image' not in request.files:
        return jsonify({'error': 'No se recibió imagen'}), 400
    img = decode_image(request.files['image'].read())
    heatmap, pred_cls, conf = grad_cam(img)

    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    axes[0].imshow(img);                           axes[0].set_title('Original');   axes[0].axis('off')
    axes[1].imshow(heatmap, cmap='jet');           axes[1].set_title('Heatmap');    axes[1].axis('off')
    overlay = 0.45 * cv2.cvtColor(
        cv2.applyColorMap(np.uint8(255*heatmap), cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB) / 255.0 + 0.55 * img
    axes[2].imshow(overlay)
    axes[2].set_title(f'Predicción: {CLASS_NAMES[pred_cls]}\n({conf*100:.0f}%)')
    axes[2].axis('off')
    plt.tight_layout()

    return jsonify({
        'predicted_class': CLASS_NAMES[pred_cls],
        'confidence':      conf,
        'visualization':   fig_to_b64(fig)
    })


@app.route('/explain/lime', methods=['POST'])
def explain_lime():
    """Retorna explicación LIME en base64 PNG."""
    if 'image' not in request.files:
        return jsonify({'error': 'No se recibió imagen'}), 400
    img     = decode_image(request.files['image'].read())
    probs   = model.predict(img[np.newaxis], verbose=0)[0]
    pred_cls = int(np.argmax(probs))

    explainer   = lime.lime_image.LimeImageExplainer(random_state=42)
    explanation = explainer.explain_instance(
        img, model.predict, top_labels=3, hide_color=0, num_samples=300)
    temp, mask = explanation.get_image_and_mask(
        pred_cls, positive_only=True, num_features=5, hide_rest=False)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(mark_boundaries(temp, mask, color=(0, 1, 0)))
    ax.set_title(f'LIME — {CLASS_NAMES[pred_cls]} ({probs[pred_cls]*100:.0f}%)')
    ax.axis('off')
    plt.tight_layout()

    return jsonify({
        'predicted_class': CLASS_NAMES[pred_cls],
        'confidence':      float(probs[pred_cls]),
        'visualization':   fig_to_b64(fig)
    })


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'CNN CIFAR-10'})


if __name__ == '__main__':
    print('🚀 Servidor XAI iniciado en http://localhost:5000')
    app.run(debug=False, host='0.0.0.0', port=5000)
'''

with open('app_flask.py', 'w', encoding='utf-8') as f:
    f.write(flask_app_code.strip())

print('✅ app_flask.py creado correctamente.')

### Paso 3 — Generación de reporte automático

In [ ]:
import datetime

def generate_report(model, history, mean_drop, global_mean,
                    comp_gcam, comp_lime):
    """Genera un reporte Markdown con los resultados del laboratorio."""
    loss, acc = model.evaluate(x_test, y_test_cat, verbose=0)
    now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

    report = f"""# Reporte — Entregable 6: Aplicación de Herramientas XAI
**Fecha de generación**: {now}  
**Dataset**: CIFAR-10  
**Modelo**: CNN (3 bloques convolucionales + BatchNorm + Dropout)

---

## 1. Rendimiento del Modelo

| Métrica | Valor |
|---------|-------|
| Loss (test) | {loss:.4f} |
| Accuracy (test) | {acc*100:.2f}% |
| Épocas entrenadas | {len(history.history['loss'])} |

## 2. Herramientas XAI Aplicadas

### LIME
- Genera explicaciones locales por superpíxeles.
- Fracción de área explicativa promedio: **{np.mean(comp_lime):.3f}**
- Resultado: regiones verdes = positivas; rojas = negativas.

### SHAP (DeepExplainer)
- Calcula contribución marginal promedio de cada píxel (valores de Shapley).
- Visualización por canal RGB; azul = impacto negativo, rojo = positivo.

### Grad-CAM
- Mapas de calor sobre la capa `conv2d_5` (última capa convolucional).
- Superposición con colormap JET sobre imagen original.

## 3. Métricas de Evaluación

| Métrica | Herramienta | Valor | Interpretación |
|---------|-------------|-------|----------------|
| Fidelidad (caída de prob.) | Grad-CAM | {mean_drop:.3f} | {'Alta' if mean_drop > 0.25 else 'Media' if mean_drop > 0.1 else 'Baja'} |
| Comprensibilidad (fracción área) | Grad-CAM | {np.mean(comp_gcam):.3f} | {'Buena' if np.mean(comp_gcam) < 0.3 else 'Regular'} |
| Comprensibilidad (fracción superpíxeles) | LIME | {np.mean(comp_lime):.3f} | {'Buena' if np.mean(comp_lime) < 0.3 else 'Regular'} |
| Estabilidad (Pearson c/ ruido) | Grad-CAM | {global_mean:.4f} | {'Alta' if global_mean > 0.85 else 'Media' if global_mean > 0.6 else 'Baja'} |

## 4. Servicio Flask

Microservicio disponible en `app_flask.py`. Endpoints:
- `GET  /health` — verificación de estado
- `POST /predict` — clasificación (multipart/form-data: imagen)
- `POST /explain/gradcam` — mapa de calor Grad-CAM
- `POST /explain/lime` — explicación LIME

## 5. Archivos generados

| Archivo | Descripción |
|---------|-------------|
| `cnn_model_cifar10.h5` | Pesos del mejor modelo CNN |
| `training_history.png` | Curvas de loss y accuracy |
| `lime_explanations.png` | Visualizaciones LIME |
| `shap_explanations.png` | Visualizaciones SHAP |
| `gradcam_explanations.png` | Mapas Grad-CAM |
| `fidelity_evaluation.png` | Gráfica de fidelidad |
| `comprehensibility_evaluation.png` | Gráfica de comprensibilidad |
| `stability_evaluation.png` | Gráfica de estabilidad |
| `app_flask.py` | Microservicio REST Flask |
| `reporte_xai.md` | Este reporte |
"""
    return report


report_text = generate_report(
    model, history, mean_drop, global_mean,
    comp_scores_gcam, comp_scores_lime_pos
)

with open('reporte_xai.md', 'w', encoding='utf-8') as f:
    f.write(report_text)

print('✅ Reporte guardado como reporte_xai.md')
print()
print(report_text)

### Test del servicio Flask (desde el notebook)
> Ejecutar en **terminal separada**: `python app_flask.py`  
> Luego correr la celda siguiente para verificar el endpoint `/health`.

In [ ]:
# Test manual del endpoint de salud
# (Requiere que app_flask.py esté corriendo en otra terminal)
try:
    import requests as req
    r = req.get('http://localhost:5000/health', timeout=3)
    print(f'✅ Servicio Flask respondió: {r.json()}')
except Exception as e:
    print(f'⚠️  Servicio no disponible: {e}')
    print('   → Ejecuta en terminal: python app_flask.py')

### Reflexión Final — Actividad 4

> El microservicio Flask permite integrar las explicaciones XAI en sistemas reales 
> (aplicaciones web, dashboards médicos, sistemas de calidad industrial) sin necesidad 
> de que el usuario final tenga conocimiento de ML. Cada endpoint retorna la predicción 
> junto con una visualización en base64, fácilmente embebible en HTML o apps móviles.
>
> **Extensiones posibles**: añadir autenticación JWT, cacheo de mapas con Redis, 
> logging estructurado y contenerización con Docker para escalar horizontalmente.

---
# 🏁 Conclusiones Generales

| Aspecto | Resultado |
|---------|----------|
| Modelo CNN | Entrenado con data augmentation; accuracy típica 70-78% en CIFAR-10 |
| LIME | Explicaciones locales intuitivas; útil para usuarios no técnicos |
| SHAP | Cuantificación rigurosa por píxel; costosa pero precisa |
| Grad-CAM | Rápida y visualmente clara; ideal para producción |
| Evaluación | Fidelidad, comprensibilidad y estabilidad medidas objetivamente |
| Flask | Microservicio REST listo para integración en producción |

### Aplicabilidad en otros contextos

- **Diagnóstico médico**: Grad-CAM puede resaltar lesiones en rayos X o ecografías.
- **Control de calidad industrial**: LIME puede identificar defectos específicos en imágenes de manufactura.
- **Seguridad (detección facial)**: SHAP puede revelar qué rasgos activan una clasificación, ayudando a detectar sesgos.
- **Conducción autónoma**: Grad-CAM muestra si el modelo mira la señal de tráfico correcta o elementos irrelevantes.

> Las herramientas XAI no solo mejoran la transparencia del modelo, sino que constituyen 
> una capa esencial de **gobernanza de IA responsable**, especialmente en sistemas críticos 
> donde las decisiones automatizadas impactan directamente a personas.